# 03 — Read staged SeaweedFS objects through S3A

**Student exercise**

Complete the tasks in order. Each task includes an expected result, three hints, and a check. All sample data and setup code are included in this notebook.

## 1. Spark configuration

Start a fresh kernel. Use Spark 3.5.x / Scala 2.12 and Java 17. Set `STUDENT_ID` to a lowercase name containing letters, numbers or underscores. Set `SPARK_MASTER=local[2]` for a local Spark session.

SeaweedFS S3 endpoint: `http://localhost:9001`. Use an existing `datalake` bucket. Setup and write tasks overwrite only the specified exercise prefix; use a distinct student ID. The driver and workers must be able to reach this endpoint. Hadoop AWS 3.3.4 must match the Hadoop version in Spark; connector JARs are downloaded from Maven.

In [ ]:
import os
import socket
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

master_url = os.environ.get("SPARK_MASTER", f"spark://{socket.gethostname()}:7077")
STUDENT_ID = "student01"
assert STUDENT_ID and STUDENT_ID[0].islower() and all(c in "abcdefghijklmnopqrstuvwxyz0123456789_" for c in STUDENT_ID)
base_path = f"s3a://datalake/exercises/{STUDENT_ID}/ex03"

spark = (
    SparkSession.builder
    .appName("Exercise-03")
    .master(master_url)
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9001")
    .config("spark.hadoop.fs.s3a.access.key", "team")
    .config("spark.hadoop.fs.s3a.secret.key", "team1234")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.redaction.regex", "(?i)secret|password|token|access[.]?key|credential")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## 2. Sample data

Run this cell, inspect the records, and use the supplied variables in the tasks.

In [ ]:
orders = spark.createDataFrame([
    (101,1,40.0,'2026-09-01'), (102,2,50.0,'2026-09-01'),
    (103,1,30.0,'2026-09-02'), (104,3,20.0,'2026-09-02')],
    'id INT, customer_id INT, amount DOUBLE, order_date STRING')
orders.show()
customers = spark.createDataFrame([(1,'Asha','India'),(2,'Ben','UK'),(3,'Carla','Brazil')],
    'customer_id INT, name STRING, country STRING')
csv_uri = f"{base_path}/input/orders"
json_uri = f"{base_path}/input/customers"
orders.write.mode("overwrite").option("header", "true").csv(csv_uri)
customers.write.mode("overwrite").json(json_uri)
print(csv_uri, json_uri)


## 3. Result checks

This small function checks output values and duplicate counts. It is included here so the notebook can run on its own.

In [ ]:
def check_frame(df, columns, expected):
    assert df is not None, "Complete the task before running the check"
    assert set(columns).issubset(df.columns), "Missing output columns"
    def normalize(row):
        return tuple(round(value, 6) if isinstance(value, float) else value for value in row)
    actual = sorted(repr(normalize(row)) for row in df.select(*columns).collect())
    wanted = sorted(repr(normalize(row)) for row in expected)
    assert actual == wanted, f"Expected {wanted}, got {actual}"
    print("PASS:", columns)

## Task 1: Read a staged CSV with an explicit schema

Read csv_uri with header=True and the given types, creating orders.

**Expected result:** Four order rows; id is integer.

In [ ]:
orders = None  # TODO: spark.read.schema(...).option('header', True).csv(csv_uri)

**Hint 1:** s3a:// identifies a bucket and key prefix; HTTP 9001 is the gateway endpoint.

**Hint 2:** Set header before csv so the header is not read as a data row.

**Hint 3:** spark.read.schema("id INT, ...").option("header", True).csv(csv_uri).

### Check

In [ ]:
check_frame(orders,['id','amount'],[(101,40.0),(102,50.0),(103,30.0),(104,20.0)])

## Task 2: Read JSON Lines

Read json_uri as JSON with customer_id INT,name STRING,country STRING. Create customers.

**Expected result:** Three customer rows.

In [ ]:
customers = None  # TODO: schema and json reader

**Hint 1:** JSON Lines has one JSON object per line; multiLine is not required.

**Hint 2:** The same S3A endpoint settings apply to both readers.

**Hint 3:** spark.read.schema(...).json(json_uri).

### Check

In [ ]:
check_frame(customers,['customer_id','name'],[(1,'Asha'),(2,'Ben'),(3,'Carla')])

## Task 3: Write and read back a derived Parquet dataset

Write orders with amount>=35 to f"{base_path}/output/high_value" using overwrite, then read it as roundtrip.

**Expected result:** Parquet retains two typed rows.

In [ ]:
roundtrip = None  # TODO: filter, write.parquet, read.parquet

**Hint 1:** Spark output is a prefix containing part files, not necessarily a single named file.

**Hint 2:** Only overwrite an output under this exercise/student prefix.

**Hint 3:** filtered.write.mode("overwrite").parquet(...); spark.read.parquet(...).

### Check

In [ ]:
check_frame(roundtrip,['id','amount'],[(101,40.0),(102,50.0)])

## More practice

Inspect inputFiles(), explain the part files and _SUCCESS marker, and compare CSV/JSON schemas.

## Common mistakes

A browser console endpoint is not the S3 API. Hadoop-AWS must match the actual Hadoop runtime. Do not print credentials.

## Finish

Stop Spark before starting another notebook. For write exercises, run setup again before repeating append tasks. S3 data remains available after stopping Spark.

In [ ]:
spark.stop()